# Lesson 1.5 — From linear regression to neural networks (and what BERT actually is)

Companion to [Lesson 1](lesson_01_linear_regression.ipynb).

After Lesson 1 you've trained a model with **2 parameters** that learns to fit a line. After Lesson 4 you train a tiny BERT with **3000 parameters** that does fill-in-the-blank. This notebook bridges the gap.

We'll answer two questions that should be on your mind:

1. **"Wait — is BERT just linear regression with more knobs? Or is it a different kind of model?"**
2. **"When do you need a neural network instead of linear regression?"**

The story we'll build today, on a single dataset:

| Approach | Parameters | Can it fit the curve? |
|---|---|---|
| **Linear regression** (L1 — `y = w*x + b`) | 2 | ❌ No |
| **Polynomial regression** (same loop, more features) | 3 | ✅ Yes (with handcrafted x² feature) |
| **Neural network** (Linear → ReLU → Linear) | ~30 | ✅ Yes (no feature engineering needed) |
| **Transformer** (BERT, PRAGMA) | thousands–billions | ✅ Yes, on sequences |

> 🔑 **The big takeaway:** the **5-line training loop is the same for all four**. What changes is the **model in the middle**. BERT and PRAGMA are NEURAL NETWORKS — specifically a kind called Transformers — NOT linear regression. The training recipe is shared; the model architecture is different.


## Step 0 — Imports

In [ ]:
import torch                                # PyTorch: tensors, autograd, optimisers.
import torch.nn as nn                       # Neural-net classes (Linear, Embedding, Module).
import torch.nn.functional as F             # Stateless ops (F.relu, F.softmax, F.mse_loss).

torch.manual_seed(0)                        # Fix RNG so every run reproduces these numbers.
torch.set_printoptions(precision=3,         # Print tensors with 3 decimals and
                       sci_mode=False)      # no scientific notation. Cleaner inline output.

## Step 1 — A new dataset: this one is a CURVE

In Lesson 1 the data was a straight line: `y = 2x + 1`. Linear regression nailed it because a line is exactly what linear regression can fit.

This time the secret rule is **a parabola**: `y = x² - 4x + 3`. Let's see if linear regression can still handle it.

In [ ]:
# Generate the data
x_data = torch.linspace(-2, 6, 40)          # 40 evenly-spaced x values from -2 to 6.
y_true = x_data ** 2 - 4 * x_data + 3 \
         + torch.randn(40) * 0.5            # Secret rule y = x² − 4x + 3 with Gaussian
                                            # noise (std=0.5) sprinkled in.

print(f"x range: {x_data.min().item():.1f} to {x_data.max().item():.1f}")   # .item() pulls float.
print(f"y range: {y_true.min().item():.1f} to {y_true.max().item():.1f}")
print()                                     # Blank line.

# Show first/last few points
print(f"{'x':>6}  {'y_true':>7}")           # Right-aligned column headers.
print("-" * 16)                             # ASCII divider.
for i in [0, 5, 10, 15, 20, 25, 30, 35, 39]:    # 9 representative indices.
    print(f"{x_data[i].item():>6.2f}  {y_true[i].item():>7.2f}")     # One row per index.

**Look at the y values.** They go DOWN and then UP. That's not a line. No straight line could possibly fit all those points well — a line either goes up the whole way or down the whole way.

Let's draw it (ASCII):

In [ ]:
def ascii_scatter(x, y, marker='*', width=60, height=20, title=""):
    # Helper that plots (x, y) points in ASCII without matplotlib.
    # width/height set plot size; marker is the character drawn.
    if title:
        print(title)
    xmin, xmax = x.min().item(), x.max().item()    # x-axis range from data.
    ymin, ymax = y.min().item(), y.max().item()    # y-axis range from data.
    grid = [[" "] * width for _ in range(height)]  # Blank 2D canvas.
    for xi, yi in zip(x.tolist(), y.tolist()):     # Iterate over each point.
        col = int((xi - xmin) / (xmax - xmin) * (width - 1))    # Map x → column.
        row = height - 1 - int((yi - ymin) / (ymax - ymin) * (height - 1))    # Map y → row (flipped).
        if 0 <= col < width and 0 <= row < height:    # Bounds check.
            grid[row][col] = marker                # Stamp marker.
    print(f"{ymax:6.1f} |" + "".join(grid[0]))     # Top row with max-y label.
    for row in grid[1:-1]:                         # Middle rows.
        print("       |" + "".join(row))
    print(f"{ymin:6.1f} |" + "".join(grid[-1]))    # Bottom row with min-y label.
    print("       +" + "-" * width)                # X-axis line.
    print(f"        x={xmin:.1f}{' ' * (width-12)}x={xmax:.1f}")    # X-axis labels.

ascii_scatter(x_data, y_true, title="The data (a parabola):")

See the U-shape? That's our challenge. Linear regression can only draw STRAIGHT lines, and there is no straight line that hits all those points.

## Step 2 — Try linear regression (will fail)

Same model as Lesson 1: `y = w*x + b`. Two parameters. Let's see how badly it fails.

In [ ]:
# Model: y = w*x + b
w = torch.tensor(0.0, requires_grad=True)   # Slope parameter, starts at 0.
                                            # requires_grad=True tells PyTorch to track it for backward().
b = torch.tensor(0.0, requires_grad=True)   # Intercept, also starts at 0.
opt = torch.optim.SGD([w, b], lr=0.001)     # SGD optimiser — "nudge each param against its gradient"
                                            # with step size lr=0.001 (small to avoid divergence).

# Train
for _ in range(1000):                       # Repeat the 5-line training loop 1000 times.
    y_pred = w * x_data + b                 # Forward pass for all 40 inputs at once (broadcasting).
    loss = ((y_pred - y_true) ** 2).mean()  # Mean squared error — one scalar measure of wrongness.
    opt.zero_grad(); loss.backward(); opt.step()    # Clear grads, backprop, nudge w and b.

print(f"Linear regression result: y = {w.item():.3f} * x + {b.item():.3f}")
print(f"Final loss: {loss.item():.3f}")     # Should be ~25 — no straight line fits a parabola.
print()
print("(Compare to L1 which got loss ≈ 0.0001 on its line.)")
print()

# Predictions
y_pred_linear = (w * x_data + b).detach()   # .detach() removes from autograd graph (just inspecting).
print("First few predictions vs truth:")
print(f"{'x':>6}  {'y_true':>7}  {'y_pred':>7}  {'error':>7}")
print("-" * 33)
for i in [0, 10, 20, 30, 39]:               # 5 representative indices spanning the data.
    err = y_pred_linear[i].item() - y_true[i].item()
    print(f"{x_data[i].item():>6.2f}  {y_true[i].item():>7.2f}  {y_pred_linear[i].item():>7.2f}  {err:>+7.2f}")

**Look at the errors.** At the endpoints (low and high x) the model's predictions are way off. The best straight line just CAN'T fit a curve.

Loss settles around ~3-5, much worse than L1's ~0.0001 on its straight-line data.

## Step 3 — Trick: give linear regression more features

Here's the key insight that confuses everyone: **"linear regression" doesn't mean "fits a line in x"**. It means **"linear in the parameters"**. We can give it any transformation of x as a feature!

If we add `x²` as a feature, the model becomes:

$$y = w_1 \cdot x + w_2 \cdot x^2 + b$$

This is still LINEAR REGRESSION (3 parameters: w₁, w₂, b). But now it can fit a parabola.

This is called **polynomial regression**, but it's mathematically just linear regression with extra features.

In [ ]:
# Same SGD, more parameters
w1 = torch.tensor(0.0, requires_grad=True)  # Coefficient of x (slope on the linear term).
w2 = torch.tensor(0.0, requires_grad=True)  # NEW: coefficient of x². This handcrafted feature
                                            # is what lets a linear model capture curvature.
b  = torch.tensor(0.0, requires_grad=True)  # Intercept (bias). Shifts the whole curve up/down.
opt = torch.optim.SGD([w1, w2, b], lr=0.0005)   # SGD over 3 params now. Smaller lr because x²
                                                 # values get large (up to 36), making gradients bigger.

# Train
for _ in range(2000):                       # 2000 steps — slightly more than the linear model needed.
    y_pred = w1 * x_data + w2 * x_data**2 + b   # Forward pass: y = w1*x + w2*x² + b.
    loss = ((y_pred - y_true) ** 2).mean()  # Same MSE loss — agnostic to model structure.
    opt.zero_grad(); loss.backward(); opt.step()    # Same 5-line training step.

print(f"Polynomial regression result:")
print(f"  y = {w1.item():.3f} * x + {w2.item():.3f} * x² + {b.item():.3f}")
print(f"  (true rule was:  y =  -4.000 * x +  1.000 * x² + 3.000)")
print()
print(f"Final loss: {loss.item():.3f}    (was ~25 for the pure linear model — a 25x improvement)")
print()
y_pred_poly = (w1 * x_data + w2 * x_data**2 + b).detach()    # Cache final predictions.
print("First few predictions vs truth:")
print(f"{'x':>6}  {'y_true':>7}  {'y_pred':>7}  {'error':>7}")
print("-" * 33)
for i in [0, 10, 20, 30, 39]:
    err = y_pred_poly[i].item() - y_true[i].item()
    print(f"{x_data[i].item():>6.2f}  {y_true[i].item():>7.2f}  {y_pred_poly[i].item():>7.2f}  {err:>+7.2f}")

**Beautiful.** Loss drops dramatically. The model recovered the true coefficients (almost) perfectly:

- True: `y = -4x + x² + 3`
- Found: `y ≈ -4x + 1x² + 3`

**Same training loop. More features. Better model.** This is still linear regression — we just gave it more knobs and more inputs.

🤔 **But wait — there's a catch.** How would you know to add `x²`? What if the relationship was `sin(x)` or something more complex?

You'd have to TRY many features (x, x², x³, sin(x), e^x...) and see which ones help. For simple data this is doable. For complex data (text, images, banking events) — there are too many possible features to enumerate.

**That's where neural networks come in.**

## Step 4 — Enter the neural network

A neural network is a **stack of linear transformations with nonlinear activations in between**.

The simplest one (a **Multilayer Perceptron**, or MLP) looks like this:

```
input x
   │
   ▼
Linear  ──► 8 hidden numbers
   │
   ▼
ReLU    ──► same 8 numbers, but negative values clipped to 0
   │
   ▼
Linear  ──► 1 output number (the prediction)
```

The key insight: **the ReLU activation introduces NONLINEARITY**. Without it, two stacked linear layers would still produce a linear function (because composing linear functions gives a linear function). With ReLU in between, the network can model curves, kinks, and arbitrarily complex patterns.

**Universal approximation theorem (don't worry about the math):** with enough hidden units, an MLP can approximate ANY smooth function. You don't have to handcraft features like `x²` — the network learns them automatically.

In [ ]:
class MLP(nn.Module):                       # Inherit from nn.Module — the base class for every
                                            # PyTorch model (gives us .parameters(), .train(), etc.).
    """A tiny neural network: 1 input → 8 hidden units → 1 output."""
    def __init__(self, hidden=8):           # Constructor. hidden = number of units in middle layer.
        super().__init__()                  # Required boilerplate to set up nn.Module internals.
        self.layer1 = nn.Linear(1, hidden)  # First linear: 1 input → 8 hidden.
                                            # Holds (8,1) weight + (8,) bias = 16 trainable knobs.
        self.layer2 = nn.Linear(hidden, 1)  # Second linear: 8 hidden → 1 output.
                                            # Holds (1,8) weight + (1,) bias = 9 more knobs.

    def forward(self, x):                   # Called whenever we do net(x). Defines the model's rule.
        h = self.layer1(x)                  # Apply first linear: h = x @ W^T + b. Shape: (batch, 8).
        h = F.relu(h)                       # THE crucial nonlinearity — clips negatives to 0.
                                            # Without this, stacking Linears collapses to one Linear.
        return self.layer2(h)               # Apply second linear. Shape: (batch, 1). The prediction.

net = MLP(hidden=8)                         # Instantiate. Weights get random init automatically.
total = sum(p.numel() for p in net.parameters())   # Count total trainable numbers.
print(f"Tiny MLP architecture:")
for name, p in net.named_parameters():      # Walk over each parameter tensor.
    shape_str = str(tuple(p.shape))         # Format shape as a string for the table.
    print(f"  {name:<20s}  shape {shape_str:<10s}  {p.numel()} params")
print(f"\nTotal parameters: {total}")
print(f"(Compare to: 2 for linear regression, 3 for polynomial.)")

### 🎨 Visualise the architecture

Before we train, let's draw the network as a wiring diagram. Knowing the SHAPE of the model up front makes it much easier to follow what's happening during training.

In [ ]:
# ASCII architecture diagram of the MLP
hidden = 8                                  # Number of hidden units (same as MLP default).
print("Neural network architecture (Linear → ReLU → Linear):")
print()
print(f"    {'INPUT':<10s}      {'HIDDEN ('+str(hidden)+' units)':<22s}   {'OUTPUT':<10s}")
print()
# Top of diagram
print(f"                          ┌───┐")
print(f"                          │h₁ │──┐")
for i in range(2, hidden):                  # Draw middle hidden units (h_2 .. h_7).
    print(f"                          ├───┤  │")
    print(f"                          │h_{i}│──┤")
print(f"                          ├───┤  │")
print(f"      x ─────────────────►│h_{hidden}│──┤───────► y")    # Main flow: x in, y out.
print(f"                          └───┘  │")
print(f"                                 │ (weighted sum)")
print()
print("       layer1: nn.Linear(1, " + str(hidden) + ")        ReLU clipping")
print("                                  layer2: nn.Linear(" + str(hidden) + ", 1)")
print()
print("Parameter breakdown:")
print(f"  layer1.weight  shape (8, 1)    8 numbers   ← one 'slope' per hidden unit")
print(f"  layer1.bias    shape (8,)      8 numbers   ← one 'kink point' per hidden unit")
print(f"  layer2.weight  shape (1, 8)    8 numbers   ← how much each hidden unit contributes")
print(f"  layer2.bias    shape (1,)      1 number    ← final offset")
print(f"                                ────────")
print(f"                                  {total} total")
print()
print("Conceptually:")
print("  Each hidden unit h_i computes:  ReLU(w_i · x + b_i)")
print("    → A 'knee' function: zero below some threshold, then linear ramp upward.")
print("    → The knee is at x = -b_i / w_i.")
print("  The 8 knees ADD UP (weighted by layer2) to make the final prediction curve.")
print("  With 8 knees you can approximate any reasonable curve.")

### 🎲 What do the weights look like at INITIALISATION (random)?

PyTorch initialises weights randomly. Let's print them. The model is currently incompetent — these numbers have no relationship to the data.

In [ ]:
print("Initial (random) weights:")
print()
print(f"  {'unit':<6s}  {'w (slope)':>11s}  {'b (bias)':>11s}  {'kink at x =':>13s}")
print("  " + "-" * 50)
for i in range(hidden):                     # Loop over each of the 8 hidden units.
    w = net.layer1.weight[i, 0].item()      # i-th unit's slope from the (8,1) weight matrix.
    b = net.layer1.bias[i].item()           # i-th unit's bias from the (8,) bias vector.
    kink = -b / w if abs(w) > 1e-6 else float('inf')    # x = -b/w is where the ReLU "turns on".
    print(f"  h_{i+1:<4d}  {w:>+11.3f}  {b:>+11.3f}  {kink:>+13.2f}")
print()
print("Layer 2 (how each hidden unit contributes to the output):")
print(f"  layer2 weights: {net.layer2.weight.detach()[0].tolist()}")
print(f"  layer2 bias:    {net.layer2.bias.item():.3f}")
print()
print("Right now these are noise. Training will reshape them so each hidden")
print("unit becomes a useful 'feature detector' for some region of x.")

## Step 5 — Train the MLP on the SAME parabola data

Same training loop. Same loss function. Same optimizer. Only the model changed.

In [ ]:
import copy                                # For deep-copying tensors when we snapshot weights.

net = MLP(hidden=8)                         # Fresh model so we start from random weights.
opt = torch.optim.Adam(net.parameters(), lr=0.05)   # Adam: adaptive learning rate per parameter.
                                            # Better than SGD when gradients vary a lot in magnitude.

# We need to reshape x_data to (N, 1) — neural nets expect a batch dimension
x_in = x_data.unsqueeze(-1)     # (40, 1)   # nn.Linear expects (batch, features). Adds feature axis.
y_target = y_true.unsqueeze(-1) # (40, 1)   # Same reshape so MSE-loss shapes match.

# Capture weight snapshots at these training steps so we can see evolution
CHECKPOINTS = [0, 100, 500, 1000, 2000]     # Steps at which we save a full snapshot.
snapshots = {}                              # step_number → snapshot dict.

# Snapshot at step 0 (random init) before any training
snapshots[0] = {
    "w1":   net.layer1.weight.detach().clone(),    # Layer-1 weight tensor. detach + clone
                                                   # = independent copy not affected by later training.
    "b1":   net.layer1.bias.detach().clone(),      # Layer-1 bias.
    "w2":   net.layer2.weight.detach().clone(),    # Layer-2 weight.
    "b2":   net.layer2.bias.detach().clone(),      # Layer-2 bias.
    "loss": F.mse_loss(net(x_in), y_target).item(),    # Loss at this snapshot.
}

history = []                                # List of (step, loss) for the curve printout below.
for step in range(1, 2001):                 # Train for 2000 steps.
    y_pred = net(x_in)                      # Forward pass — calls MLP.forward().
    loss = F.mse_loss(y_pred, y_target)     # Same MSE loss.
    opt.zero_grad(); loss.backward(); opt.step()    # Standard 3-line training step.
    if step in CHECKPOINTS:                 # Save a snapshot at our chosen checkpoints.
        snapshots[step] = {
            "w1":   net.layer1.weight.detach().clone(),
            "b1":   net.layer1.bias.detach().clone(),
            "w2":   net.layer2.weight.detach().clone(),
            "b2":   net.layer2.bias.detach().clone(),
            "loss": loss.item(),
        }
    if step % 200 == 0:                     # Also record loss every 200 steps for the curve.
        history.append((step, loss.item()))

print(f"{'step':>5}  {'loss':>8}")
print("-" * 16)
for step, l in history:
    print(f"{step:>5}  {l:>8.4f}")

# Get final predictions
y_pred_nn = net(x_in).squeeze(-1).detach()  # Cache (40,)-shape predictions for plotting later.

## Step 5b — Watch the neural network EVOLVE during training (Karpathy-style)

This is the cool part. We saved snapshots of all the weights at 5 checkpoints during training. Let's visualise how each hidden unit's "knee" moves and how the prediction curve takes shape.

For each checkpoint, we'll show TWO things:

1. **Layer 1 weights as a small grid** — see how each unit's slope and bias change.
2. **Per-unit response curve** — for each of the 8 hidden units, plot `ReLU(w·x + b)` across the input range. Each unit becomes a feature detector for some region of x.

In [ ]:
def render_response_curves(snap, x_range, width=40):
    # For each hidden unit, render its ReLU output across x_range as a bar.
    w1 = snap["w1"]                         # (8, 1) layer-1 weights from snapshot.
    b1 = snap["b1"]                         # (8,) layer-1 biases from snapshot.
    x = x_range.unsqueeze(0)                # Reshape to (1, 40) for broadcasting against w1.
    responses = F.relu(w1 * x + b1.unsqueeze(-1))   # Compute ReLU(w*x + b) per (unit, x). Shape (8, 40).
    out = []
    for i in range(responses.size(0)):      # Process each of the 8 hidden units.
        row = responses[i]
        if row.max().item() < 1e-3:         # Unit is dead — never fires.
            out.append("." * width)
            continue
        bars = ""
        normed = row / row.max()            # Normalise to [0,1] for the bar heights.
        for v in normed.tolist():
            ch = " ▁▂▃▄▅▆▇█"[min(int(v * 8), 8)]    # Map [0,1] → block char.
            bars += ch
        out.append(bars)
    return out

# Make a dense x-range for the response plots
x_viz = torch.linspace(-2, 6, 40)

for step in CHECKPOINTS:                    # Loop over our 5 saved snapshots.
    snap = snapshots[step]
    print("=" * 70)
    print(f"  CHECKPOINT step {step}   (loss = {snap['loss']:.4f})")
    print("=" * 70)
    print()
    print(f"  {'unit':<5s}  {'w':>8s}  {'b':>8s}  {'kink at x':>10s}  {'layer2_w':>10s}")
    print("  " + "-" * 50)
    w1 = snap["w1"]; b1 = snap["b1"]; w2 = snap["w2"]   # Unpack tensors locally.
    for i in range(hidden):
        wv = w1[i, 0].item()                # Slope.
        bv = b1[i].item()                   # Bias.
        kink = -bv / wv if abs(wv) > 1e-6 else float('nan')     # Kink point at x = -b/w.
        contribution = w2[0, i].item()      # Layer-2 weight for this unit's contribution.
        kink_str = f"{kink:+.2f}" if abs(kink) < 100 else "  inf"
        print(f"  h_{i+1:<3d}  {wv:>+8.3f}  {bv:>+8.3f}  {kink_str:>10s}  {contribution:>+10.3f}")
    print()
    print(f"  Hidden unit response curves across x = [{-2}, {6}]:")
    curves = render_response_curves(snap, x_viz)
    for i, row in enumerate(curves):
        print(f"  h_{i+1}: {row}")
    print()

**Read the evolution top-to-bottom.**

- **Step 0** — all 8 hidden units have random "knees". Their response curves look noisy. The model knows nothing.
- **Step 100** — units are starting to specialise. Some have learned to respond to negative x, some to positive x.
- **Step 500** — the curves are clearly differentiated. Each unit responds to a particular region.
- **Step 2000 (final)** — beautiful "knee" functions. Each hidden unit covers a different slice of x, and together (weighted by layer 2) they sum to recreate the parabola.

This is what "training" actually looks like inside a neural network: **the weights drift from noise to structure**, where each unit becomes a feature detector for some part of the input. The same thing happens in massive networks like BERT and GPT — just with billions of weights instead of 25.

> 🔑 **The Karpathy moment:** if you watch the response curves develop, you can SEE the model thinking. Each hidden unit is finding a slot — a section of the input it's responsible for. That's machine learning made visible.

### Putting it together: how do 8 knee-shapes become a parabola?

The final prediction is `y = sum(layer2_w[i] * h_i(x)) + layer2_b`. Each hidden unit contributes a piece (positive or negative). The 8 contributions sum to the prediction curve.

In [25]:
print("Final hidden-unit contributions across the x range:")
print("  (Each row is one hidden unit's contribution: layer2_w[i] * ReLU(layer1_w[i]*x + layer1_b[i]))")
print()
print(f"  {'x':>6s} | " + " ".join(f"h{i+1:>5d}" for i in range(hidden)) + f"  | {'SUM':>7s}  {'TRUTH':>7s}")
print("  " + "-" * (10 + hidden*7 + 22))

final_snap = snapshots[2000]                # Use the final trained weights.
w1, b1 = final_snap["w1"], final_snap["b1"]
w2, b2 = final_snap["w2"], final_snap["b2"]

for i in [0, 5, 10, 15, 20, 25, 30, 35, 39]:
    x = x_data[i].item()
    contribs = []
    for j in range(hidden):                 # Walk over the 8 hidden units.
        h_j = max(0.0, w1[j, 0].item() * x + b1[j].item())    # ReLU(w*x + b) by hand.
        c   = w2[0, j].item() * h_j         # This unit's signed contribution.
        contribs.append(c)
    total = sum(contribs) + b2.item()       # Sum + layer-2 bias = full prediction.
    truth = y_true[i].item()
    print(f"  {x:>6.2f} | " + " ".join(f"{c:>+5.1f}" for c in contribs) + f"  | {total:>+7.2f}  {truth:>+7.2f}")
print()
print("Each column is one hidden unit's contribution at each input.")
print("Adding ACROSS each row reconstructs the prediction. That's all a neural net does.")

Final hidden-unit contributions across the x range:
  (Each row is one hidden unit's contribution: layer2_w[i] * ReLU(layer1_w[i]*x + layer1_b[i]))



TypeError: only integer tensors of a single element can be converted to an index

**Loss converges similarly to the polynomial regression** — both end up around the noise floor (~0.25 due to the random noise we added).

The MLP figured out the parabolic shape **without being told to add x² as a feature**. It learned the right features on its own.

## Step 6 — Side-by-side comparison

All three models on the same data:

In [ ]:
print(f"{'x':>6}  {'y_true':>7}  {'linear':>7}  {'poly':>7}  {'NN':>7}")
print("-" * 40)
for i in [0, 5, 10, 15, 20, 25, 30, 35, 39]:
    print(f"{x_data[i].item():>6.2f}  {y_true[i].item():>7.2f}  "
          f"{y_pred_linear[i].item():>7.2f}  "
          f"{y_pred_poly[i].item():>7.2f}  "
          f"{y_pred_nn[i].item():>7.2f}")
print()
loss_linear = ((y_pred_linear - y_true) ** 2).mean().item()     # MSE of the linear model.
loss_poly   = ((y_pred_poly   - y_true) ** 2).mean().item()     # MSE of the polynomial.
loss_nn     = ((y_pred_nn     - y_true) ** 2).mean().item()     # MSE of the neural net.
print(f"Final MSE: linear={loss_linear:.3f}   polynomial={loss_poly:.3f}   neural net={loss_nn:.3f}")

**The linear model fails. The polynomial and the neural network both succeed.**

But notice: the polynomial uses **3 parameters**, the neural net uses **25**. The neural net is using extra capacity that it didn't strictly need for this simple problem.

So why ever use a neural network? Because for **complex data** (text, images, banking event sequences), you don't know what the right features are. You can't say "add x²" because the data is too complex for that intuition to apply. A neural network with enough capacity will figure out the features for you.

## Step 7 — Visualise all three models' fits

ASCII overlay so you can see them on the data:

In [ ]:
def ascii_overlay(x, y_true, predictions, markers, width=70, height=18):
    # Overlay multiple prediction curves on the same ASCII plot.
    xmin, xmax = x.min().item(), x.max().item()
    all_y = torch.cat([y_true] + [p for p in predictions])    # SHARED y-axis range.
    ymin, ymax = all_y.min().item(), all_y.max().item()

    grid = [[' '] * width for _ in range(height)]   # Blank canvas.
    # Plot truth as '.'
    for xi, yi in zip(x.tolist(), y_true.tolist()):
        col = int((xi - xmin) / (xmax - xmin) * (width - 1))
        row = height - 1 - int((yi - ymin) / (ymax - ymin) * (height - 1))
        if 0 <= col < width and 0 <= row < height:
            grid[row][col] = '·'
    # Plot each prediction
    for pred, marker in zip(predictions, markers):
        for xi, yi in zip(x.tolist(), pred.tolist()):
            col = int((xi - xmin) / (xmax - xmin) * (width - 1))
            row = height - 1 - int((yi - ymin) / (ymax - ymin) * (height - 1))
            if 0 <= col < width and 0 <= row < height and grid[row][col] == ' ':
                grid[row][col] = marker     # Don't overwrite truth dots.

    print(f"{ymax:6.1f} |" + "".join(grid[0]))
    for row in grid[1:-1]:
        print("       |" + "".join(row))
    print(f"{ymin:6.1f} |" + "".join(grid[-1]))
    print("       +" + "-" * width)
    print(f"       Legend:  ·  = true data  L = linear  P = polynomial  N = neural net")

print("All three models' predictions overlaid on the data:")
ascii_overlay(x_data, y_true, [y_pred_linear, y_pred_poly, y_pred_nn], ['L', 'P', 'N'])

**Read the plot:**
- The `·` dots are the true data (the parabola).
- The `L`s trace a straight line — linear regression can ONLY draw lines.
- The `P`s follow the parabola — polynomial regression nails it.
- The `N`s also follow the parabola — neural network also nails it.

Both `P` and `N` succeed. The `L` is doomed.

## Step 7.5 — Heads-up: you just built the MLP that lives inside every Transformer

Before we get to the BERT reveal, there's something important to point out.

The MLP you trained in Steps 4-5 — `Linear → ReLU → Linear` — has a fancy name when it shows up inside a Transformer: the **feed-forward sub-layer** (sometimes "FFN" for short, or "MLP block" interchangeably).

You'll hear all of these in the literature:

- "MLP" — Multi-Layer Perceptron, the generic term you've used in this notebook
- "Feed-forward network" or "FFN" — what people call this in Transformer papers
- "MLP block" — what library docs (PyTorch, HuggingFace) often call it
- "Position-wise feed-forward" — the verbose phrasing that emphasises it's applied per-token

**They all mean the same thing: Linear → nonlinearity → Linear.**

This matters because **every Transformer block (BERT, GPT, PRAGMA, ChatGPT) contains one** of these. You've just trained an instance of it. The feed-forward sub-layer isn't an exotic new concept — it's literally the network you've been staring at for the past few cells.

### How BERT's feed-forward differs from our MLP

Two tiny differences. Architecturally identical otherwise.

In [ ]:
# Our MLP from earlier in this notebook — for comparison
class OurMLP(nn.Module):
    def __init__(self, hidden=8):
        super().__init__()
        self.layer1 = nn.Linear(1, hidden)      # 1 input → 8 hidden.
        self.layer2 = nn.Linear(hidden, 1)      # 8 hidden → 1 output.
    def forward(self, x):
        return self.layer2(F.relu(self.layer1(x)))      # Linear → ReLU → Linear.


# How BERT writes the SAME architecture, just bigger and with GELU
class BertFeedForward(nn.Module):
    def __init__(self, d_model=512):            # d_model = embedding dimension (token vector size).
        super().__init__()
        self.up   = nn.Linear(d_model, 4 * d_model)     # Expand: d → 4d. Standard 4× ratio.
        self.down = nn.Linear(4 * d_model, d_model)     # Contract: 4d → d. Back to original size.
    def forward(self, x):
        return self.down(F.gelu(self.up(x)))    # Linear → GELU → Linear. Same shape as our MLP!


# Side-by-side comparison
print(f"{'feature':<30s}  {'our MLP (L1.5)':<22s}  {'BERT feed-forward':<22s}")
print("-" * 80)
print(f"{'Architecture':<30s}  Linear → ReLU → Linear   Linear → GELU → Linear")
print(f"{'Input dim':<30s}  1                       d_model (e.g. 512)")
print(f"{'Hidden dim':<30s}  8                       4 × d_model (e.g. 2048)")
print(f"{'Output dim':<30s}  1                       d_model")
print(f"{'Activation':<30s}  ReLU                    GELU (smoother ReLU variant)")
print()
print("That's it. Two differences:")
print("  1. SIZE — BERT's is way bigger (2048 hidden vs 8). More capacity.")
print("  2. ACTIVATION — GELU instead of ReLU. Smoother. Trains slightly better.")
print()
print("Otherwise it's the EXACT SAME ARCHITECTURE you just trained.")
print("A real Transformer has ~18 of these stacked. You built one of them today.")

**The takeaway:** when you read "feed-forward layer" or "FFN" or "MLP block" in any paper about Transformers, just picture the small MLP you trained on the parabola data. Same thing, scaled up.

This also means you can now name the **three core building blocks** of any Transformer:

1. **Embedding** — Lesson 2. Turn tokens into vectors.
2. **Attention** — Lesson 3. Let every token look at every other token.
3. **Feed-forward** — THIS lesson. Apply a small MLP to each token's vector.

That's it. Stack 18 copies of (attention + feed-forward), add an output head, and you have BERT/GPT/PRAGMA.

## Step 7.6 — What does "stacking layers" mean? Why does BERT use 18 layers and not 2?

You've trained an MLP with **one** hidden layer. BERT stacks **18** "blocks", where each block is attention + feed-forward (your MLP). What does that buy us?

**The intuition** — each layer takes the output of the previous layer and computes new features from it. So:

- Layer 1 looks at the raw input and finds basic patterns
- Layer 2 looks at Layer 1's patterns and combines them into more complex patterns
- Layer 18 has access to deeply abstracted features built across 17 prior rounds

For very complex data (text, images, banking events), this **hierarchical** processing is what lets the model learn rich patterns. Simple-data tasks like the parabola don't need depth. But the parabola was easy — let's try a harder function and see depth pay off.

### A harder function: a sine wave

A sine wave wiggles. To fit it with our knee-based MLP, each "knee" can only cover a section between two zero-crossings. With 2 periods of sine on x ∈ [−3, 3], you need roughly 4-8 well-placed knees to capture it well.

In [ ]:
# A new dataset: sin(2x) on x ∈ [-3, 3] — two full periods, hard to fit with a 1-layer MLP.
x_hard = torch.linspace(-3, 3, 80)              # 80 inputs.
y_hard = torch.sin(2 * x_hard) + torch.randn(80) * 0.05     # Sine wave + small noise.

print(f"x range: {x_hard.min().item():.1f} to {x_hard.max().item():.1f}")
print(f"y range: {y_hard.min().item():.2f} to {y_hard.max().item():.2f}")
print()

ascii_scatter(x_hard, y_hard, title="The new harder dataset (a sine wave):")

### Define a configurable-depth MLP

Same MLP idea as before, but with a `n_layers` parameter so we can try multiple depths in a loop.

In [ ]:
class DeepMLP(nn.Module):
    """MLP with configurable depth. n_layers = number of hidden layers."""
    def __init__(self, hidden=16, n_layers=1):  # Default: 16 hidden units, 1 hidden layer.
        super().__init__()
        layers = []                              # Will be a list of Linear/ReLU modules.
        layers.append(nn.Linear(1, hidden))      # Input layer: 1 → hidden.
        layers.append(nn.ReLU())                 # First nonlinearity.
        for _ in range(n_layers - 1):            # Add more hidden layers if requested.
            layers.append(nn.Linear(hidden, hidden))   # hidden → hidden.
            layers.append(nn.ReLU())             # Nonlinearity between each pair.
        layers.append(nn.Linear(hidden, 1))      # Output layer: hidden → 1.
        self.net = nn.Sequential(*layers)        # Bundle into one Sequential module.

    def forward(self, x):
        return self.net(x)                       # Run all layers in order.

# Verify the parameter counts at different depths
for n in [1, 2, 4]:
    m = DeepMLP(hidden=16, n_layers=n)
    total = sum(p.numel() for p in m.parameters())
    print(f"  DeepMLP(hidden=16, n_layers={n}): {total} parameters")

### Train models at depths 1, 2, 4 on the sine wave

Same number of training steps for each. Same hidden width (16). The only thing changing is **depth**.

In [ ]:
x_in   = x_hard.unsqueeze(-1)               # Shape (80, 1).
y_targ = y_hard.unsqueeze(-1)               # Shape (80, 1).

results = []                                # Collect final losses per depth.
for n_layers in [1, 2, 4]:
    torch.manual_seed(42)                   # Same init seed for fair comparison.
    deep = DeepMLP(hidden=16, n_layers=n_layers)    # Use `deep` (not `net`) to avoid
                                                    # clobbering the earlier MLP `net`.
    opt = torch.optim.Adam(deep.parameters(), lr=0.01)
    final_loss = None
    for step in range(2000):                # Train 2000 steps each.
        loss = F.mse_loss(deep(x_in), y_targ)
        opt.zero_grad(); loss.backward(); opt.step()
        final_loss = loss.item()
    results.append((n_layers, final_loss, sum(p.numel() for p in deep.parameters())))

print(f"{'depth':>7s}  {'params':>7s}  {'final loss':>11s}")
print("-" * 32)
for depth, loss, params in results:
    print(f"{depth:>7d}  {params:>7d}  {loss:>11.5f}")

**Read the loss column.** Going deeper drops the loss — sometimes by a lot. Why?

The 1-layer MLP can only do `Linear → ReLU → Linear`. That's a single round of "create some features → combine them". It has 16 "knees" to place across x ∈ [-3, 3]. That's barely enough to trace a 2-period sine wave.

The 4-layer MLP does FOUR rounds of feature-then-combine. Each round can build on the previous one. The first round might learn rough zero-crossings; the second sharpens them; the third smooths the curves; the fourth fine-tunes. Each layer gets to **transform the features the previous layer found**, not just the raw input.

> 🔑 **Depth lets the model build hierarchical features.** A 1-layer model can only see linear combinations of input + one nonlinearity. A deeper model can compose nonlinearities — see neighbors-of-neighbors, and so on.


### Why this matters for Transformers

**Important clarification first.** A common confusion: *"Does attention happen only in the first layer, or in every layer?"*

**Answer: attention happens in EVERY single layer.** A Transformer block is an (attention + feed-forward) pair, and we duplicate that ENTIRE pair 18 times. Not just the feed-forward part. Not just the attention part. The WHOLE block.

Here's the stack drawn out:

```
    input tokens
        │
        ▼
    ┌──────────────┐
    │  Embedding   │   ← Lesson 2. Happens ONCE at the bottom.
    └──────────────┘
        │
        ▼
    ┌──────────────┐                                ┐
    │  Attention   │ ◄── tokens mix across          │
    ├──────────────┤                                │  BLOCK 1
    │ Feed-forward │ ◄── each token's own MLP       │
    └──────────────┘                                ┘
        │
        ▼
    ┌──────────────┐                                ┐
    │  Attention   │ ◄── tokens mix AGAIN           │
    ├──────────────┤                                │  BLOCK 2
    │ Feed-forward │ ◄── each token's MLP AGAIN     │
    └──────────────┘                                ┘
        │
        ▼
       ... (16 more blocks, each one is attention + FFN) ...
        │
        ▼
    ┌──────────────┐                                ┐
    │  Attention   │                                │  BLOCK 18
    ├──────────────┤                                │
    │ Feed-forward │                                │
    └──────────────┘                                ┘
        │
        ▼
    ┌──────────────┐
    │ Output head  │   ← Predict the masked token.
    └──────────────┘
```

So when someone says *"BERT is 18 layers deep"* they mean **18 attention rounds AND 18 feed-forward rounds**, interleaved. The embedding only happens once (at the bottom). The output head only happens once (at the top). Everything in between repeats.

**Why this matters intuitively** — each block builds on the previous block's output:

- **Block 1** — each token looks at its neighbours and absorbs simple context. ("my left neighbour is 'the'.")
- **Block 2** — each token (which now has Block 1's context-aware representation) looks at OTHER context-aware tokens. So it sees *neighbours-of-neighbours*. ("my left neighbour is 'the', whose left neighbour was 'see'.")
- **Block 3** — three-hop context.
- ...
- **Block 18** — each token has 18 successive rounds of attention behind it. Its vector now encodes very long-range, hierarchical patterns — far beyond what a single attention round could capture.

This is exactly like the depth experiment you just ran on the sine wave: a 1-layer MLP could only do one round of (Linear → ReLU → Linear). The 4-layer version did four rounds and got much lower loss. **Transformers benefit from depth for the same reason**, except each "round" is now an attention + feed-forward pair instead of just Linear + ReLU.

For a 2-token "dog bark" task, 1 or 2 blocks is plenty. For a 120-token banking event history with subtle behavioural patterns — Revolut's PRAGMA-L paper finds 18 blocks worth the investment.

> 🧠 **The practical rule.** For toy datasets: depth doesn't help much. For real datasets (text, images, banking events): depth is essential. That's why we use ~6–18 layers for production Transformers — and **every one of those layers includes its own attention step**.


## Step 7.7 — Embeddings and attention work in BOTH frameworks

Before you head to Lessons 2 and 3, here's a subtle but important point. So far we've sorted models into two camps:

- **Linear regression / polynomial regression** — no nonlinearity. Just `y = Σ wᵢ · feature_i + b`.
- **Neural networks** — at least one nonlinearity (ReLU, GELU, softmax, etc.).

**Embeddings and attention are NOT model classes.** They're *building blocks* that can be plugged into either kind of model. You can use an embedding inside a linear-regression-style head, or inside a neural-net head. Same for attention.

This is important because it tells you what each block is actually contributing:

| Block | Adds nonlinearity? | What it adds |
|---|---|---|
| **Embedding** | No (pure lookup → no curve) | Trainable vector per discrete input |
| **Attention** | Yes (softmax inside) | Cross-token mixing |
| **Feed-forward** | Yes (GELU inside) | Per-token nonlinear capacity |

So an embedding by itself is closer to "linear regression with one-hot inputs"; the softmax inside attention and the GELU inside feed-forward are where the real nonlinearity comes from. Let's see this concretely.


### Demo A — Embedding plugged into a LINEAR-REGRESSION-style model

Toy task: predict the typical noise level of an animal (numeric output) from the animal's name (categorical input, 5 categories).

A pure linear model needs numeric inputs. We don't have that — we have categorical names. So we'll give each animal an **embedding** (a small learnable vector) and feed it into a linear head: `y = w · embedding + b`. Still linear regression in spirit — no ReLU, no GELU, no softmax. The embedding is the ONLY new piece.


In [ ]:
torch.manual_seed(0)                            # Repeatable randomness.

# Tiny categorical dataset: animal id → typical noise level.
animals = ["mouse", "cat", "dog", "lion", "elephant"]   # 5 animal names (our vocabulary).
noise_y = torch.tensor([0.2, 1.0, 2.5, 4.5, 6.0])       # Target loudness for each animal.

ids = torch.arange(5)                           # Integer ids 0..4, one per animal.

# Embedding = learnable vector per animal id. 5 animals × 3-dim vectors.
emb = nn.Embedding(num_embeddings=5,            # Vocabulary size.
                   embedding_dim=3)             # Each animal becomes a length-3 vector.

# Linear-regression-style head: y = w · embedding + b. NO nonlinearity.
head_linear = nn.Linear(3, 1)                   # 3 embedding dims → 1 output. Linear only.

params = list(emb.parameters()) + list(head_linear.parameters())   # All trainable knobs.
opt = torch.optim.Adam(params, lr=0.05)         # Same Adam, same loop, same MSE loss.

for step in range(800):                         # Standard 5-line training loop.
    vec  = emb(ids)                             # Look up the 5 embedding vectors. Shape (5, 3).
    pred = head_linear(vec).squeeze(-1)         # Linear projection to (5,). NO activation.
    loss = F.mse_loss(pred, noise_y)            # Same MSE as Lesson 1.
    opt.zero_grad(); loss.backward(); opt.step()

print(f"Final loss (linear-style head): {loss.item():.4f}")
print()
print(f"{'animal':<10s}  {'true y':>7s}  {'pred y':>7s}  {'embedding (length 3)':<30s}")
print("-" * 60)
final_vecs = emb(ids).detach()                  # Snapshot final embeddings for inspection.
final_pred = head_linear(emb(ids)).squeeze(-1).detach()
for name, y, p, v in zip(animals, noise_y, final_pred, final_vecs):
    v_str = "[" + ", ".join(f"{x:+.2f}" for x in v.tolist()) + "]"
    print(f"{name:<10s}  {y.item():>7.2f}  {p.item():>7.2f}  {v_str}")

**Read the result.** Loss should be tiny (~0). Embeddings + a linear head perfectly fit the target noise levels because, with 3 free dimensions per animal, there's plenty of room for `w · emb[i] + b` to hit any number.

This is **linear regression with a learnable feature lookup**. The embedding gave a numeric vector to a discrete input. The head was still pure linear. Same Lesson 1 idea, just with embeddings.

### Demo B — Same embedding, now plugged into a NEURAL-NET head

What if we swap the linear head for an MLP (Linear → ReLU → Linear) like in Step 4? The embedding doesn't change. Only the head class changes.


In [ ]:
torch.manual_seed(0)                            # Same seed, fair comparison.

emb_nn = nn.Embedding(5, 3)                     # Same shape: 5 animals × 3 dims.

# Neural-net head: Linear → ReLU → Linear. Same MLP idea as Step 4.
head_nn = nn.Sequential(
    nn.Linear(3, 8),                            # 3-dim embedding → 8 hidden units.
    nn.ReLU(),                                  # Nonlinearity.
    nn.Linear(8, 1),                            # 8 hidden → 1 output.
)

params = list(emb_nn.parameters()) + list(head_nn.parameters())
opt = torch.optim.Adam(params, lr=0.05)

for step in range(800):                         # Identical training loop.
    vec  = emb_nn(ids)                          # Same embedding lookup. Shape (5, 3).
    pred = head_nn(vec).squeeze(-1)             # Now goes through a NONLINEAR head.
    loss = F.mse_loss(pred, noise_y)
    opt.zero_grad(); loss.backward(); opt.step()

print(f"Final loss (neural-net head):   {loss.item():.4f}")
print()
print("Conclusion: the embedding works the SAME WAY in both cases.")
print("It's just a lookup table of trainable vectors. What you feed those")
print("vectors INTO (linear head vs MLP head) is a separate decision.")

### Demo C — Attention plugged into a LINEAR-REGRESSION-style model

Now the same point for attention. Task: given a tiny 3-token sequence, predict a single number — the **maximum value's index** times some coefficient. To do this well the model must let tokens "look at each other" (so each one knows whether it's the max). That's exactly what attention does.

We'll use a single attention layer + a **linear** head — no MLP, no ReLU. The softmax inside attention IS a nonlinearity, but everything we add ourselves is linear.


In [ ]:
torch.manual_seed(0)

# Tiny sequence dataset: (3-token sequence) → scalar target = sum of the tokens.
# A pure linear-per-token model can't do this without cross-token mixing.
N = 64                                          # Number of training sequences.
seqs = torch.rand(N, 3, 1)                      # Shape (batch, seq=3, features=1).
tgt  = seqs.sum(dim=1).squeeze(-1)              # Sum across the 3 tokens. Shape (N,).

# Attention (with softmax inside) + LINEAR head only.
attn        = nn.MultiheadAttention(embed_dim=1,        # 1 feature per token.
                                    num_heads=1,        # Single head for simplicity.
                                    batch_first=True)
head_linear = nn.Linear(3, 1)                   # Linear over the 3 attended outputs.

params = list(attn.parameters()) + list(head_linear.parameters())
opt = torch.optim.Adam(params, lr=0.05)

for step in range(600):
    mixed, _ = attn(seqs, seqs, seqs)           # Self-attention: tokens look at each other.
                                                # The softmax inside IS the only nonlinearity.
    flat   = mixed.squeeze(-1)                  # (N, 3) — 3 mixed scalars per sequence.
    pred   = head_linear(flat).squeeze(-1)      # Linear projection to a single number.
    loss   = F.mse_loss(pred, tgt)
    opt.zero_grad(); loss.backward(); opt.step()

print(f"Attention + LINEAR head, final loss: {loss.item():.4f}")
print()
print("First few predictions:")
print(f"{'sequence':<28s}  {'true':>6s}  {'pred':>6s}")
print("-" * 50)
for i in range(5):
    s = "[" + ", ".join(f"{x.item():.2f}" for x in seqs[i].squeeze(-1)) + "]"
    print(f"{s:<28s}  {tgt[i].item():>6.2f}  {pred[i].item():>6.2f}")

### Demo D — Same attention, now followed by a NEURAL-NET head

Same attention layer, but instead of a single Linear afterwards we use Linear → ReLU → Linear (an MLP). This is **exactly** the (attention + feed-forward) pair that a Transformer block uses, in miniature.


In [ ]:
torch.manual_seed(0)

attn_nn = nn.MultiheadAttention(1, 1, batch_first=True)     # Identical attention layer.
head_nn = nn.Sequential(                        # Now a real MLP head.
    nn.Linear(3, 16),                           # 3 mixed scalars → 16 hidden units.
    nn.ReLU(),                                  # Nonlinearity (in addition to softmax in attn).
    nn.Linear(16, 1),                           # 16 hidden → 1 prediction.
)

params = list(attn_nn.parameters()) + list(head_nn.parameters())
opt = torch.optim.Adam(params, lr=0.05)

for step in range(600):
    mixed, _ = attn_nn(seqs, seqs, seqs)        # Same attention call.
    flat     = mixed.squeeze(-1)                # (N, 3)
    pred     = head_nn(flat).squeeze(-1)        # Now through an MLP head.
    loss     = F.mse_loss(pred, tgt)
    opt.zero_grad(); loss.backward(); opt.step()

print(f"Attention + MLP head, final loss:    {loss.item():.4f}")
print()
print("Conclusion: attention works the SAME WAY in both cases.")
print("It mixes tokens across positions. What you put on top of it")
print("(linear head vs MLP head) is a separate choice — and both work.")
print()
print("A real Transformer block uses the MLP option (Demo D), and stacks")
print("18 copies of it. That's the whole architecture.")

### Recap: model class vs building block

You just trained **four** models. Two used embeddings, two used attention. Two used linear heads, two used MLP heads. Mix and match.

| Demo | Building block | Head class | Worked? |
|---|---|---|---|
| A | Embedding | Linear (linear regression style) | ✅ |
| B | Embedding | MLP (neural net style) | ✅ |
| C | Attention | Linear (linear regression style) | ✅ |
| D | Attention | MLP (neural net style) | ✅ |

**The key insight:** *embeddings* and *attention* are reusable building blocks. The "linear vs neural" decision is a separate axis — it's about whether the **head** that consumes their output has a nonlinearity.

When you read Lesson 2, you'll be looking at an embedding in isolation. When you read Lesson 3, you'll be looking at attention in isolation. Both lessons happen to combine their building block with an MLP head, because that's what a real Transformer does. But now you know: **those blocks would still work if the head were purely linear**. The block itself is the new idea; the head is just how we hook it up to the loss.

> 🔑 **Mental model for Lessons 2 → 5:** Embedding turns tokens into vectors. Attention mixes the vectors across positions. Feed-forward (MLP) processes the vectors per position. The Transformer stacks (attention + feed-forward) eighteen times, each block's output feeding the next. Same 5-line training loop the whole time.


## Step 8 — The big reveal: what IS BERT, then?

Now we can answer the question that may have been confusing.

**BERT, GPT, PRAGMA — they are NEURAL NETWORKS.** Specifically, a kind called **Transformers**. And a Transformer is built out of THREE pieces — all of which you now know from earlier lessons:

```
input tokens
   │
   ▼
[ Embedding ]    ─►  Lesson 2.  Turn each token into a vector.
   │                            (Linear lookup. No nonlinearity here.)
   ▼
[ Attention ]    ─►  Lesson 3.  Every vector looks at every other vector.
   │                            (Q/K/V projections + softmax. Nonlinear via softmax.)
   ▼
[ Feed-forward ] ─►  THIS lesson — the MLP you just trained!
   │                            (Linear → GELU → Linear. Nonlinear via GELU.)
   ▼
... (stack 18 copies of attention + feed-forward) ...
   │
   ▼
[ Output head ]  ─►  Linear projection to vocabulary scores. Predict the masked token.
```

**You know all three pieces.** This is not a black box. You've trained instances of each one:

- Lesson 2 showed you what an embedding is (a lookup table of learnable vectors)
- Lesson 3 showed you what attention is (Q · K → softmax → weighted V)
- This lesson showed you what feed-forward is (Linear → ReLU → Linear, with a few thousand parameters per layer in BERT)

The "trick" of the Transformer is just **interleaving attention and feed-forward** many times. Attention mixes information ACROSS tokens; feed-forward processes information WITHIN each token. Together they give the model enough capacity to learn rich patterns from sequences.

The key part: **nonlinearities are everywhere** (softmax inside attention, GELU inside feed-forward). Without them, stacking would collapse into one big linear function — useless. Same reason the MLP needs ReLU between its Linears.

**So no — BERT is not linear regression. BERT is a neural network made of attention + feed-forward, repeated.**

What's the same across L1 and BERT:
- ✅ The 5-line training loop (predict → loss → backward → step)
- ✅ Gradient descent (Adam or SGD)
- ✅ Trainable parameters (the "knobs" — just lots more of them in BERT)

What's different:
- ❌ The middle: linear regression has `w*x + b`; BERT has 18 stacks of attention + feed-forward.

Same training. Different model.

## Step 9 — The complete picture: model family tree

| Model | Architecture | Parameters | Where it shines |
|---|---|---|---|
| **Linear regression** (L1) | `y = w*x + b` | 2 | Predicting a single number from a single number, when the relationship is linear |
| **Polynomial regression** (L1.5 above) | `y = w₁x + w₂x² + ... + b` | 3 to N | Same, when relationship is polynomial and you know what features to add |
| **MLP / Neural net** (L1.5 above) | Linear → ReLU → Linear → (...) | tens to thousands | Predicting any function of any feature vector, when the features can be learned |
| **CNN** (not in this course) | Convolutions + ReLU + pooling | millions | Images |
| **RNN / LSTM** (briefly in L3b) | Recurrent steps over a sequence | millions | Sequences (but with memory issues — see L3b) |
| **Transformer** (L4, L5, BERT, PRAGMA, GPT) | **Embedding + Attention + Feed-forward** (stacked) | millions to **trillions** | Sequences, where every token can look at every other token |

**Every single one is trained with the SAME 5-line loop.** Only the model class changes.

> 🧠 **The three pieces of a Transformer, summarised:**
>
> | Piece | Lesson | What it does | What it adds |
> |---|---|---|---|
> | **Embedding** | L2 | Token → vector | Turns words into math |
> | **Attention** | L3 | Each vector looks at every other vector | Cross-token mixing |
> | **Feed-forward (MLP)** | L1.5 (this!) | Each vector → bigger vector → smaller vector | Per-token capacity / nonlinearity |
>
> If you understand these three, you understand the Transformer. The rest is just stacking, residual connections, and LayerNorm (small engineering tweaks that make stacking work).

## Step 10 — Inspect what the neural net "learned"

Just for fun: peek inside the trained MLP and see what each hidden unit responds to.

In [ ]:
# Run the trained MLP and look at the hidden activations
with torch.no_grad():                       # Disable autograd inside this block — saves memory.
    hidden = F.relu(net.layer1(x_in))       # Compute layer-1 + ReLU. Shape (40, 8).

print("Hidden unit activations across the input range:")
print(f"  Each row = one input x; each column = one of the 8 hidden units")
print(f"  '·' means 0 (ReLU clipped it); '█' means active")
print()
print(f"  {'x':>6s} | " + " ".join(f"h{i}" for i in range(8)))
print("  " + "-" * 30)
for i in [0, 5, 10, 15, 20, 25, 30, 35, 39]:
    row = ""
    for h in hidden[i].tolist():
        row += " " + ("█" if h > 0.5 else ("▄" if h > 0.01 else "·"))    # 3-level char.
    print(f"  {x_data[i].item():>6.2f} |{row}")
print()
print("Each hidden unit has learned to respond to a different part of the x range.")
print("Some fire for negative x, some for positive x, some for the middle.")
print("Adding them up (with the weights learned in layer2) reconstructs the parabola.")

## Step 11 — Things to try

### 🟢 Easy

1. **Different curves.** Replace `y_true = x² - 4x + 3` with `y_true = torch.sin(x_data)` (or any other curve you want). Re-train all three models. Polynomial regression will fail unless you add more features; the neural network will succeed automatically.

2. **More hidden units.** Change `MLP(hidden=8)` to `MLP(hidden=64)`. Does it train better? Worse? Faster? Slower?

### 🟡 Medium

3. **Deeper network.** Add another hidden layer:
   ```python
   self.layer1 = nn.Linear(1, 8)
   self.layer2 = nn.Linear(8, 8)
   self.layer3 = nn.Linear(8, 1)
   def forward(self, x):
       h = F.relu(self.layer1(x))
       h = F.relu(self.layer2(h))
       return self.layer3(h)
   ```
   What does deeper buy you? (On this simple problem, not much. On real problems, depth is essential.)

4. **Replace ReLU with Sigmoid or Tanh.** Try `torch.sigmoid` or `torch.tanh` instead of `F.relu`. Compare training curves.

### 🔴 Hard

5. **What happens with NO activation function?** Remove the `F.relu` line. Now the MLP collapses to a single linear function. Train it and verify it does no better than linear regression — this is the "without nonlinearity, depth doesn't help" lesson.

6. **A "neural network" with attention.** Replace the MLP's hidden layer with a `nn.MultiheadAttention` layer. (You'll need to reshape your input to `(batch, seq=1, features)`.) This is the smallest possible Transformer-like model. Train it and verify it can also fit the parabola — though attention is wildly overkill for a 1D scalar regression.


## Summary

You now know:

- ✅ Linear regression fits **lines** (only).
- ✅ Polynomial regression (still "linear regression" mathematically) fits **polynomials** if you handcraft x² and x³ as features.
- ✅ A neural network with a nonlinearity (ReLU, GELU, etc.) can fit **any smooth function** without needing handcrafted features — it learns them.
- ✅ **BERT, PRAGMA, GPT are neural networks** (specifically Transformers). Their power comes from learnable embeddings + many attention layers + nonlinearities. They are NOT linear regression.
- ✅ The **5-line training loop is the same** for all of these. What changes is the model class.

Open [Lesson 4](lesson_04_tiny_bert.ipynb) again with fresh eyes — that tiny BERT is a neural network with about 3000 parameters that uses attention. Everything before that lesson was warm-up.
